# Lab06: Writing Data in Neo4j

Santiago Elí Jiménez Aguilar
Luis Eduardo Gonzalez Gloria

## Goal:
Create a data pipeline to analyze the Recommendation Videogames dataset <br>
(https://networkrepository.com/rec-amz-Video-Games.phpLinks to an external site.).

## Instructions
- **Dataset**. Download the Recommendation Videogames dataset from the Network Repository.
- **Data Ingestion**. This section should contain a code cell using PySpark to read the DataFrame.
- **Graph Analysis**. This section should contain the code to generate:
- - PageRank
- - Label Propagation
- - Triangle Counting
- - Degree Distribution
- **Writing Data in Neo4j**. This section should contain the code to persist the nodes and edges DataFrames in Neo4j.
- **Querying the Graph**. This section should contain a screenshot of the graph written to Neo4j.

In [1]:
from spark_utils import SparkUtils
neo4j_connector = "org.neo4j:neo4j-connector-apache-spark_2.13:5.3.10_for_spark_3,io.graphframes:graphframes-spark3_2.13:0.9.0-spark3.5"
su = SparkUtils("Lab06: Writing Data in Neo4j", "spark://spark-master:7077", spark_packages=neo4j_connector)
su.spark

:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.3.jar!/org/apache/ivy/core/settings/ivysettings.xml
Ivy Default Cache set to: /root/.ivy2.5.2/cache
The jars for the packages stored in: /root/.ivy2.5.2/jars
org.neo4j#neo4j-connector-apache-spark_2.13 added as a dependency
io.graphframes#graphframes-spark3_2.13 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c56fcd19-ec2d-4c30-b085-56bbdc7cfa1f;1.0
	confs: [default]
	found org.neo4j#neo4j-connector-apache-spark_2.13;5.3.10_for_spark_3 in central
	found org.neo4j#neo4j-connector-apache-spark_2.13_common;5.3.10_for_spark_3 in central
	found org.neo4j#caniuse-core;1.3.0 in central
	found org.neo4j#caniuse-api;1.3.0 in central
	found org.jetbrains.kotlin#kotlin-stdlib;2.1.20 in central
	found org.jetbrains#annotations;13.0 in central
	found org.neo4j#caniuse-neo4j-detection;1.3.0 in central
	found org.neo4j.driver#neo4j-java-driver-slim;4.4.21 in central
	found org.reactivestreams#reactiv

## Create GraphFrames

In [2]:
from graphframes import GraphFrame
from pyspark.sql import functions as F

# ✅ Definir el schema primero
mvideo_games_schema = SparkUtils.generate_schema([
    ("userId",      "string"),
    ("videoGameId", "string"),
    ("rating",      "float"),
    ("timestamp",   "long")
])

# Ahora sí puedes leer el archivo
video_games_df = (su.spark.read 
                .option("header", "false")   # ← el archivo NO tiene encabezado
                .schema(mvideo_games_schema)
                .csv("/opt/spark/work-dir/data/rec-amz-Video-Games"))

# 1. VÉRTICES - usuarios únicos
user_vertices = video_games_df.select(
    F.col("userId").alias("id"),
    F.lit("user").alias("type")
).distinct()

# 2. VÉRTICES - videojuegos únicos
game_vertices = video_games_df.select(
    F.col("videoGameId").alias("id"),
    F.lit("game").alias("type")
).distinct()

# 3. Unir ambos tipos de vértices
vertices = user_vertices.union(game_vertices)

# 4. ARISTAS
edges = video_games_df.select(
    F.col("userId").alias("src"),
    F.col("videoGameId").alias("dst"),
    F.col("rating").alias("rating"),
    F.col("timestamp").alias("timestamp")
)

# 5. Crear el grafo
g = GraphFrame(vertices, edges)
g.vertices.show()
g.edges.show()

/usr/local/lib/python3.10/dist-packages/pyspark/sql/classic/dataframe.py:146: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(
                                                                                

+--------------+----+
|            id|type|
+--------------+----+
| AE7GUHCDQQ4UI|user|
|A26B0P6K95SIKW|user|
|A182S3ANC0W7DL|user|
|A1T98OCCYW6OBI|user|
|A1TBUSGCBTXWFC|user|
|A366EUKI8WMGYB|user|
|A129SW886TYQ6H|user|
|A2M64UKVOU9CWU|user|
|A10N7L0GMRODUO|user|
| AFWPLXT2OD6H1|user|
|A2890J1FHE76RY|user|
| AJRHPTQ7TXPD6|user|
|A2NQXA21O64HDZ|user|
|A261CI99KET69W|user|
|A214Z566V6QEDF|user|
|A1NC9PQCE3NOGA|user|
|A27QRTHZLBA61M|user|
| ADOCLYEFV2PKH|user|
| A8SSL0QMV2VY1|user|
| AA9LU15A9PX9E|user|
+--------------+----+
only showing top 20 rows
+--------------+----------+------+----------+
|           src|       dst|rating| timestamp|
+--------------+----------+------+----------+
| AB9S9279OZ3QO|0078764343|   5.0|1373155200|
|A24SSUT5CSW8BH|0078764343|   5.0|1377302400|
| AK3V0HEBJMQ7J|0078764343|   4.0|1372896000|
|A10BECPH7W8HM7|043933702X|   5.0|1404950400|
|A2PRV9OULX1TWP|043933702X|   5.0|1386115200|
| AE7GUHCDQQ4UI|043933702X|   1.0|1366156800|
| A48ABFDDRMKI8|043933702X|   5.0

## Core Graph Algorithms
### PageRank

In [3]:
# ── PageRank ──────────────────────────────────────────────────────────────────
results = g.pageRank(resetProbability=0.15, maxIter=10)

# ── Top usuarios más "importantes" (muy conectados / bien valorados) ──────────
print("=== TOP USUARIOS POR PAGERANK ===")
results.vertices \
    .filter(F.col("type") == "user") \
    .select("id", "type", "pagerank") \
    .orderBy("pagerank", ascending=False) \
    .show(10)

# ── Top videojuegos más "importantes" ────────────────────────────────────────
print("=== TOP VIDEOJUEGOS POR PAGERANK ===")
results.vertices \
    .filter(F.col("type") == "game") \
    .select("id", "type", "pagerank") \
    .orderBy("pagerank", ascending=False) \
    .show(10)

# ── Aristas con sus pesos calculados ─────────────────────────────────────────
print("=== ARISTAS CON PESO ===")
results.edges \
    .select("src", "dst", "rating", "weight") \
    .orderBy("weight", ascending=False) \
    .show(10)

/usr/local/lib/python3.10/dist-packages/pyspark/sql/classic/dataframe.py:128: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


=== TOP USUARIOS POR PAGERANK ===


+--------------------+----+------------------+
|                  id|type|          pagerank|
+--------------------+----+------------------+
|                 ...|user|0.5551520502498027|
|                 ...|user|0.5551520502498027|
|        <h3 class...|user|0.5551520502498027|
|A00063061AK7XBIZL...|user|0.5551520502498027|
|    <div class="p...|user|0.5551520502498027|
|A0009878M2RGMMHGJH39|user|0.5551520502498027|
|        <h4>Pleas...|user|0.5551520502498027|
|A00278362652PLQL0...|user|0.5551520502498027|
|             </html>|user|0.5551520502498027|
|A00338543M2OZPUWO...|user|0.5551520502498027|
+--------------------+----+------------------+
only showing top 10 rows
=== TOP VIDEOJUEGOS POR PAGERANK ===


+----------+----+------------------+
|        id|type|          pagerank|
+----------+----+------------------+
|B00DJFIMW6|game| 7285.632480821969|
|B00BGA9WK2|game|2645.5457035600543|
|B00FAX6XQC|game|2547.3712122892407|
|B009KS4XRO|game| 2501.074994509743|
|B0055SWM08|game| 1989.485284686411|
|B00CSR2J9I|game| 1964.401877042414|
|B002VBWIP6|game| 1857.220477824996|
|B0015AARJI|game|1408.3899898091515|
|B000FKBCX4|game|1243.5735678537487|
|B00178630A|game|1230.2925647140119|
+----------+----+------------------+
only showing top 10 rows
=== ARISTAS CON PESO ===


[Stage 664:======================================>                  (2 + 1) / 3]

+--------------------+----------+------+------+
|                 src|       dst|rating|weight|
+--------------------+----------+------+------+
|A0122991ZW10PWNGS11Z|B0050SZ836|   1.0|   1.0|
|A0163098YTSQ38GSG1VL|B00BGHUS58|   4.0|   1.0|
|A0065174S9P164B537UU|B001EYU1VO|   5.0|   1.0|
|A0381358EQKNFUTM8IJR|B002I0J4VQ|   1.0|   1.0|
|A02049035W9C30B2NBJM|B00DJFIMW6|   5.0|   1.0|
|A01877783UI8HDN5V...|B0057S9JQ6|   5.0|   1.0|
|A01638641K2WLM3WG...|B004MKN3YE|   5.0|   1.0|
|A01882789KR2FROBMN8K|B008L3UUPS|   1.0|   1.0|
|A012367930ASU72P8...|B00B8CNLD2|   1.0|   1.0|
|A001147626R4BL248...|B00BXONG7G|   2.0|   1.0|
+--------------------+----------+------+------+
only showing top 10 rows


## Label Propagation

In [4]:
lpa = g.labelPropagation(maxIter=5)
lpa.show()

[Stage 852:=============================================>           (4 + 1) / 5]

+--------------------+----+-----------+
|                  id|type|      label|
+--------------------+----+-----------+
|                 ...|user|17179869214|
|    <div class="p...|user|          4|
|          0439339960|game| 8589986060|
|          0439394422|game|34359774680|
|          0439591368|game|25769922868|
|          0439900581|game|25769878021|
|          1886846758|game|17179911578|
|          6050036071|game|     136706|
|          7118021156|game| 8589957042|
|          9078439122|game|34359858185|
|          9572132148|game|25769858567|
|          9573499126|game|17180023684|
|          9755334602|game|        640|
|          9861767304|game| 8589947394|
|          9882151809|game|25769809030|
|             </html>|user|         38|
|<div class="col-m...|user|         39|
|              <html>|user|         40|
|A0002090WKEMAO8KOWKM|user|17180042394|
|A00096001PYDTZQQ4...|user|25769978036|
+--------------------+----+-----------+
only showing top 20 rows


## Triangle counting

In [5]:
triangle_count = g.triangleCount()
triangle_count.show()

[Stage 989:=============================================>           (4 + 1) / 5]

+-----+--------------+----+
|count|            id|type|
+-----+--------------+----+
|    0| AE7GUHCDQQ4UI|user|
|    0|A1TBUSGCBTXWFC|user|
|    0|A129SW886TYQ6H|user|
|    0| AFWPLXT2OD6H1|user|
|    0|A26B0P6K95SIKW|user|
|    0|A366EUKI8WMGYB|user|
|    0|A10N7L0GMRODUO|user|
|    0|A261CI99KET69W|user|
|    0|A214Z566V6QEDF|user|
|    0|A182S3ANC0W7DL|user|
|    0|A2M64UKVOU9CWU|user|
|    0|A2890J1FHE76RY|user|
|    0| AJRHPTQ7TXPD6|user|
|    0|A1NC9PQCE3NOGA|user|
|    0|A27QRTHZLBA61M|user|
|    0| AA9LU15A9PX9E|user|
|    0|A1T98OCCYW6OBI|user|
|    0|A2NQXA21O64HDZ|user|
|    0| ADOCLYEFV2PKH|user|
|    0| A8SSL0QMV2VY1|user|
+-----+--------------+----+
only showing top 20 rows


# Degrees Distribution

### InDregree

In [6]:
in_deg = g.inDegrees.join(vertices, "id")
in_deg.show()

+----------+--------+----+
|        id|inDegree|type|
+----------+--------+----+
|0439394422|       2|game|
|7118021156|       2|game|
|986325083X|       1|game|
|9882077153|       7|game|
|B000006OWT|      11|game|
|B000006RGR|      28|game|
|B00000DMAI|      19|game|
|B00000I1BK|      44|game|
|B00000IFKW|       3|game|
|B00000IGZM|      13|game|
|B00000K13F|       1|game|
|B00000K2XJ|      26|game|
|B00000K4E7|       4|game|
|B00000K4KF|      26|game|
|B00000K4YE|       4|game|
|B00000K516|       5|game|
|B00000K51C|      10|game|
|B00001L5TD|       3|game|
|B00001N2MM|       1|game|
|B00001OX3R|       2|game|
+----------+--------+----+
only showing top 20 rows


### OutDegree

In [7]:
out_deg = g.outDegrees.join(vertices, "id")
out_deg.show()

[Stage 1015:============================>                           (2 + 1) / 4]

+--------------------+---------+----+
|                  id|outDegree|type|
+--------------------+---------+----+
|                 ...|        1|user|
|                 ...|        1|user|
|                 ...|        1|user|
|                 ...|        1|user|
|                 ...|        1|user|
|        <h3 class...|        1|user|
|        <h4>Pleas...|        1|user|
|    <div class="p...|        1|user|
|    <div class="p...|        1|user|
|             </html>|        1|user|
|<div class="col-m...|        1|user|
|              <html>|        1|user|
|<link rel="styles...|        1|user|
|<link rel="styles...|        1|user|
|A0002090WKEMAO8KOWKM|        1|user|
|A00063061AK7XBIZL...|        1|user|
|A00089163LKXK4V19...|        1|user|
|A00096001PYDTZQQ4...|        2|user|
|A0009878M2RGMMHGJH39|        1|user|
|A00101847G3FJTWYGNQA|        3|user|
+--------------------+---------+----+
only showing top 20 rows


## Write data to a Neo4j Graph

In [8]:
neo4j_url = "bolt://neo4j-iteso:7687"
neo4j_user = "neo4j"
neo4j_passwd = "neo4j@1234"

g.vertices.write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("labels", ":User") \
  .option("node.keys", "id") \
  .save()

print(f"{g.vertices.count()} verticess wrote in Neo4j")


g.edges.write \
  .format("org.neo4j.spark.DataSource") \
  .mode("Overwrite") \
  .option("url", neo4j_url) \
  .option("authentication.basic.username", neo4j_user) \
  .option("authentication.basic.password", neo4j_passwd) \
  .option("relationship", "FOLLOWS") \
  .option("relationship.save.strategy", "keys") \
  .option("relationship.source.labels", ":User") \
  .option("relationship.source.save.mode", "match") \
  .option("relationship.source.node.keys", "src:id") \
  .option("relationship.target.labels", ":User") \
  .option("relationship.target.save.mode", "match") \
  .option("relationship.target.node.keys", "dst:id") \
  .save()

print(f"{g.edges.count()} edges wrote in Neo4j")

ERROR:root:KeyboardInterrupt while sending command.                 (0 + 1) / 4]
Traceback (most recent call last):
  File "/usr/local/lib/python3.10/dist-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/usr/local/lib/python3.10/dist-packages/py4j/clientserver.py", line 535, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/usr/lib/python3.10/socket.py", line 705, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt
26/03/27 21:34:44 ERROR OverwriteByExpressionExec: Data source write support org.neo4j.spark.writer.Neo4jBatchWriter@7e8e7e0c is aborting.
26/03/27 21:34:44 ERROR OverwriteByExpressionExec: Data source write support org.neo4j.spark.writer.Neo4jBatchWriter@7e8e7e0c aborted.


KeyboardInterrupt: 

In [ ]:
su.spark.stop()